# 09 — Análisis jerárquico de agarres

**Pregunta:** ¿qué agarres se parecen fisiológicamente entre sí y cómo se organizan en familias?

La metodología es fija y el dataset es intercambiable. Funciona igual con una fila por sujeto × agarre o
con varias (ventanas como filas).

1. **El sujeto es la unidad de análisis.** Cada sujeto pesa lo mismo, tenga las ventanas que tenga.
2. **Centrado por sujeto (obligatorio).** Se elimina el nivel propio de cada sujeto y, con él, el de su
   base de datos, porque cada sujeto pertenece a una sola DB. Se comprueba con una descomposición de
   varianza que DB y sujeto quedan en 0 %.
3. Se calcula el **centroide** de cada agarre y la distancia entre centroides con **Mahalanobis**
   (covarianza dentro de los agarres, **Ledoit-Wolf**) o su versión insesgada **crossnobis**.
4. Se construye un **dendrograma** de agarres y se mide su **estabilidad** con un bootstrap por sujetos
   (estratificado por base de datos).

Las figuras **no se muestran en el notebook**: se guardan como PDF en `figures/jerarquico/<dataset>/`
(con la fecha de generación en la esquina), igual que en los notebooks 07 y 08. No se guardan
tablas: los resultados numéricos se muestran en el notebook.

### Cómo reutilizarlo con otro dataset
Cambia `DATA_PATH` (y opcionalmente `FEATURE_COLS`) en la primera celda de código.
Si `FEATURE_COLS = None`, las columnas se detectan automáticamente: todas las numéricas
que no sean metadatos (`stimulus`, `subject`, `origin_db`, ventanas, etc.).

| Sección | Contenido | Figuras |
|---|---|---|
| 1 | Carga (con detección automática de features y diseño completo) | — |
| 2 | Efecto sujeto y base de datos: antes y después del centrado | — |
| 3 | Covarianza Ledoit-Wolf y distancia entre centroides | `distancia_heatmap.pdf` |
| 4 | Dendrograma y elección del método de enlace | — |
| 5 | Estabilidad: bootstrap por sujetos | `dendrograma.pdf` |
| 6 | Sensibilidad: Mahalanobis vs crossnobis | — |
| 7 | Resumen | — |
| 8 | Interpretación | — |

In [21]:
from _provenance import Sello
sello = Sello('09_clustering_analysis.ipynb')

In [22]:
# Cambia SOLO estas líneas para probar otro dataset.
DATA_PATH = '../data/master/df_stats_wnd_mean.csv'
FEATURE_COLS = None          # None = detecta automáticamente las columnas numéricas no-meta

DISTANCIA = 'crossnobis'     # 'mahalanobis' (clásica, sesgada hacia arriba) | 'crossnobis' (insesgada)

In [23]:
# ---------------- Configuración (normalmente no se toca) ----------------
LABEL_COL, SUBJECT_COL, DB_COL = 'stimulus', 'subject', 'origin_db'
META_COLS = {LABEL_COL, SUBJECT_COL, DB_COL, 'window', 'window_index', 'window_size_samples',
             'window_size_ms', 'repetition'}
REST = 0
LINKAGE_METHODS = ['average', 'complete', 'ward']   # en caso de empate gana el primero de la lista
N_FOLDS = 5                                          # folds de sujetos para crossnobis
N_BOOT = 500                                         # remuestreos de sujetos (sección 6)
RANDOM_STATE = 42
# -------------------------------------------------------------------------

import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.covariance import LedoitWolf
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, to_tree
from scipy.spatial.distance import squareform

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

if DISTANCIA not in ('mahalanobis', 'crossnobis'):
    raise ValueError(f"DISTANCIA debe ser 'mahalanobis' o 'crossnobis', no {DISTANCIA!r}")
if DISTANCIA == 'crossnobis' and 'ward' in LINKAGE_METHODS:
    # Ward supone distancias euclídeas; crossnobis no lo es (puede incluso dar d² < 0)
    LINKAGE_METHODS = [m for m in LINKAGE_METHODS if m != 'ward']

DIST_LABEL = {'mahalanobis': 'Distancia de Mahalanobis',
              'crossnobis': 'Distancia crossnobis (Mahalanobis validada)'}[DISTANCIA]

# Fecha de generación de las figuras (sello en la esquina de cada PDF)
GEN_DATE = datetime.now().strftime('%Y-%m-%d %H:%M')


def style_ax(ax):
    ax.set_facecolor('#FFFFFF')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=8)


def savefig(fig, name):
    """Guarda la figura en PDF dentro de SAVE_DIR y libera la memoria (sin plt.show)."""
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    fig.savefig(f'{SAVE_DIR}/{name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)

In [24]:
# ---------- Funciones de análisis ----------

def cargar_datos(data_path, feature_cols=None):
    """Lee el CSV, detecta features, quita el reposo, los NaN y los sujetos con agarres faltantes
    (el centrado por sujeto exige el diseño completo)."""
    df = pd.read_csv(data_path)
    missing = [c for c in [LABEL_COL, SUBJECT_COL, DB_COL] if c not in df.columns]
    if missing:
        raise KeyError(f'Faltan columnas en el CSV ({len(missing)}): {missing}')

    if feature_cols is None:
        feature_cols = [c for c in df.columns
                        if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]
        if not feature_cols:
            raise KeyError('No se detectó ninguna columna numérica de features.')
    else:
        missing_f = [c for c in feature_cols if c not in df.columns]
        if missing_f:
            raise KeyError(f'FEATURE_COLS no está en el CSV ({len(missing_f)}): {missing_f[:10]}')

    n_rest = int((df[LABEL_COL] == REST).sum())
    df = df[df[LABEL_COL] != REST]
    n_nan = int(df[feature_cols].isna().any(axis=1).sum())
    df = df.dropna(subset=feature_cols).copy()
    df['group'] = df[SUBJECT_COL].astype(str) + '_' + df[DB_COL].astype(str)

    n_stim = df[LABEL_COL].nunique()
    stim_por_suj = df.groupby('group')[LABEL_COL].nunique()
    incompletos = stim_por_suj[stim_por_suj < n_stim].index.tolist()
    df = df[~df['group'].isin(incompletos)].reset_index(drop=True)

    info = {'feature_cols': list(feature_cols), 'n_rest': n_rest, 'n_nan': n_nan,
            'incompletos': incompletos}
    return df, info


def cell_means(X, groups, y):
    """Media de cada celda sujeto × agarre.

    Devuelve (M, g_cell, y_cell, codes): M[c] es la media de la celda c, g_cell / y_cell su sujeto y
    agarre, y codes la celda de cada fila. Con una fila por celda, M es X reordenada.
    """
    codes, _ = pd.factorize(pd.Series(groups, dtype=str) + '|' + pd.Series(y).astype(str))
    n = codes.max() + 1
    sums = np.zeros((n, X.shape[1]))
    np.add.at(sums, codes, X)
    counts = np.bincount(codes, minlength=n)
    first = np.unique(codes, return_index=True)[1]
    return sums / counts[:, None], np.asarray(groups)[first], np.asarray(y)[first], codes


def class_centroids(X, y, groups):
    """Centroide de cada agarre = media sobre sujetos de sus medias por celda (cada sujeto pesa 1)."""
    M, _, yc, _ = cell_means(X, groups, y)
    return np.vstack([M[yc == s].mean(axis=0) for s in STIMULI])


def centrar_por_sujeto(X, groups, y):
    """Resta a cada fila la media de su sujeto, calculada como la media de sus medias por agarre
    (los agarres con más ventanas no pesan más). Elimina el efecto sujeto y, con él, el de su DB."""
    M, gc, _, _ = cell_means(X, groups, y)
    subj_mean = pd.DataFrame(M).groupby(gc).mean()
    return X - subj_mean.loc[np.asarray(groups)].values


def descomponer_varianza(X, groups, y):
    """% de la varianza de cada variable (sobre las medias por celda) que es de DB, de sujeto
    dentro de su DB, de agarre y residual (interacción sujeto × agarre)."""
    M, gc, yc, _ = cell_means(X, groups, y)
    A = pd.DataFrame(M, columns=FEATURE_COLS)
    dbc = np.array([DB_OF_GROUP[g] for g in gc])
    grand = A.mean()
    tot = ((A - grand) ** 2).mean()
    m_db = A.groupby(dbc).transform('mean')
    m_sj = A.groupby(gc).transform('mean')
    m_st = A.groupby(yc).transform('mean')
    out = pd.DataFrame({
        'DB': ((m_db - grand) ** 2).mean() / tot,
        'Sujeto (dentro de DB)': ((m_sj - m_db) ** 2).mean() / tot,
        'Agarre': ((m_st - grand) ** 2).mean() / tot,
    })
    out['Residual (sujeto × agarre)'] = 1 - out.sum(axis=1)
    return 100 * out


def pooled_lw(X, y, groups):
    """Covarianza dentro de los agarres (residuos de cada fila respecto al centroide de su agarre)
    con Ledoit-Wolf. Con ventanas como filas es la covarianza del ruido ventana a ventana."""
    resid = X - class_centroids(X, y, groups)[np.searchsorted(STIMULI, y)]
    return LedoitWolf().fit(resid)


def mahalanobis_matrix(centroids, precision):
    diff = centroids[:, None, :] - centroids[None, :, :]
    d2 = np.einsum('ijk,kl,ijl->ij', diff, precision, diff)
    np.fill_diagonal(d2, 0)
    return d2


def asignar_folds(groups, n_folds, seed):
    """Fold fijo para cada sujeto (por su ID original): todas sus ventanas y sus copias del
    bootstrap quedan en el mismo fold."""
    uniq = np.random.default_rng(seed).permutation(np.unique(groups))
    return {g: i % n_folds for i, g in enumerate(uniq)}


def crossnobis_matrix(X, y, groups, folds, precision):
    """d² insesgada: producto de diferencias de centroides de folds de sujetos DISTINTOS.

    En la Mahalanobis clásica el ruido de los centroides se eleva al cuadrado y siempre suma
    (E[d²] = d² real + sesgo > 0). Con folds independientes el ruido no correlaciona y se cancela.
    """
    diffs = []
    for f in np.unique(folds):
        m = folds == f
        if len(np.unique(y[m])) < N_STIM:
            raise ValueError(f'El fold {f} no contiene todos los agarres')
        c = class_centroids(X[m], y[m], groups[m])
        diffs.append(c[:, None, :] - c[None, :, :])
    d2, n = np.zeros((N_STIM, N_STIM)), 0
    for i in range(len(diffs)):
        for j in range(i + 1, len(diffs)):
            d2 += np.einsum('abk,kl,abl->ab', diffs[i], precision, diffs[j])
            n += 1
    d2 /= n
    np.fill_diagonal(d2, 0)
    return d2


def distancia(X, y, groups, folds, metodo=None):
    """Matriz de distancias entre centroides. Devuelve (D, d², ajuste Ledoit-Wolf).

    groups = ID de sujeto de cada fila (en el bootstrap, único por copia);
    folds = fold de cada fila (por el ID original del sujeto)."""
    metodo = metodo or DISTANCIA
    lw = pooled_lw(X, y, groups)
    if metodo == 'mahalanobis':
        d2 = mahalanobis_matrix(class_centroids(X, y, groups), lw.precision_)
    else:
        d2 = crossnobis_matrix(X, y, groups, folds, lw.precision_)
    D = np.sqrt(np.clip(d2, 0, None))   # crossnobis puede dar d² < 0 (distancia real ≈ 0)
    return D, d2, lw


def elegir_enlace(D, methods):
    """Correlación cofenética por enlace; empate (±0.001) → gana el primero de la lista."""
    cond = squareform(D, checks=False)
    coph = {m: cophenet(linkage(cond, method=m), cond)[0] for m in methods}
    best_val = max(coph.values())
    empatados = [m for m in methods if coph[m] >= best_val - 1e-3]
    return coph, empatados[0], empatados


def clades(Z, n):
    """Conjuntos de hojas de cada rama interna (sin la raíz)."""
    root = to_tree(Z)
    out = []

    def walk(node):
        if node.is_leaf():
            return frozenset([node.id])
        s = walk(node.left) | walk(node.right)
        if len(s) < n:
            out.append(s)
        return s
    walk(root)
    return set(out)


def bootstrap_sujetos(Xd, y, groups, folds, Z_ref, n_boot=N_BOOT, seed=RANDOM_STATE):
    """Remuestrea sujetos con reemplazo DENTRO de cada base de datos y repite todo el proceso.
    Cada copia de un sujeto cuenta como un sujeto distinto (ID 'g#k'), pero conserva su fold."""
    rng = np.random.default_rng(seed)
    groups_by_db = {}
    for g, db in DB_OF_GROUP.items():
        groups_by_db.setdefault(db, []).append(g)
    groups_by_db = {db: np.array(gs) for db, gs in groups_by_db.items()}
    idx_by_group = {g: np.where(groups == g)[0] for g in DB_OF_GROUP}

    ref_clades = clades(Z_ref, N_STIM)
    hits = {c: 0 for c in ref_clades}
    D_boot, n_ok = [], 0
    for _ in range(n_boot):
        sample = np.concatenate([rng.choice(gs, size=len(gs), replace=True)
                                 for gs in groups_by_db.values()])
        idx = np.concatenate([idx_by_group[g] for g in sample])
        gb = np.concatenate([np.full(len(idx_by_group[g]), f'{g}#{k}', dtype=object)
                             for k, g in enumerate(sample)])
        Xb, yb, fb = Xd[idx], y[idx], folds[idx]
        try:
            Db, _, _ = distancia(Xb, yb, gb, fb)
        except ValueError:      # algún fold de crossnobis se quedó sin sujetos
            continue
        Zb = linkage(squareform(Db, checks=False), method=LINKAGE)
        for c in clades(Zb, N_STIM) & ref_clades:
            hits[c] += 1
        D_boot.append(Db)
        n_ok += 1

    support = {c: 100 * h / n_ok for c, h in hits.items()}
    return support, np.array(D_boot), n_ok

In [25]:
# ---------- Funciones de gráficas (ven los datos como globals del dataset) ----------

def plot_dist_heatmap(D_df, order):
    D_ord = D_df.loc[order, order]
    fig, ax = plt.subplots(figsize=(0.9 * N_STIM + 3, 0.8 * N_STIM + 1.5), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    sns.heatmap(D_ord, annot=True, fmt='.2f', cmap='mako_r', mask=np.eye(N_STIM, dtype=bool),
                linewidths=0.5, linecolor='#FFFFFF', cbar_kws={'label': DIST_LABEL}, ax=ax)
    ax.grid(False)
    ax.set_title(f'{DIST_LABEL} entre centroides\n(ordenada según el dendrograma)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    savefig(fig, 'distancia_heatmap')


def plot_dendrograma(Z, support, n_ok):
    fig, ax = plt.subplots(figsize=(11, 6), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    dn = dendrogram(Z, labels=LABELS, leaf_font_size=11, color_threshold=0,
                    above_threshold_color='#37474F', ax=ax)

    # Conjunto de hojas de cada unión
    node_sets = {}
    for i, (a, b, _, _) in enumerate(Z):
        def members(k):
            return {int(k)} if k < N_STIM else node_sets[int(k) - N_STIM]
        node_sets[i] = members(a) | members(b)

    # Posición horizontal: hojas en 5, 15, 25…; cada unión en el punto medio de sus hijos
    node_x = {LABELS.index(lbl): 5 + 10 * k for k, lbl in enumerate(dn['ivl'])}
    for i, (a, b, _, _) in enumerate(Z):
        node_x[N_STIM + i] = (node_x[int(a)] + node_x[int(b)]) / 2

    for i, (_, _, h, _) in enumerate(Z):
        s = frozenset(node_sets[i])
        if len(s) == N_STIM:
            continue
        sup = support.get(s, 0)
        color = '#2E7D32' if sup >= 70 else ('#F9A825' if sup >= 50 else '#C62828')
        ax.text(node_x[N_STIM + i], h, f'{sup:.0f}%', ha='center', va='bottom', fontsize=8,
                fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=color, lw=0.8))

    ylabel = 'Altura de unión (criterio de Ward)' if LINKAGE == 'ward' else DIST_LABEL
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.set_title(f'Dendrograma de agarres — {DISTANCIA} (Ledoit-Wolf, centrado por sujeto) + enlace '
                 f'{LINKAGE}\n% = soporte bootstrap por sujetos ({n_ok} remuestreos; verde ≥ 70 %, '
                 f'rojo < 50 %)', fontweight='bold', fontsize=11)
    style_ax(ax)
    ax.tick_params(axis='x', labelsize=10)
    plt.tight_layout()
    savefig(fig, 'dendrograma')


## 1. Carga

El dataset ya viene transformado: **no se vuelve a escalar**. Al cargar:
- Se detectan las features (todas las columnas numéricas que no son metadatos, si `FEATURE_COLS = None`).
- Se quitan el reposo (si existe) y las filas con NaN.
- Se excluyen los **sujetos a los que les falta algún agarre**: el centrado por sujeto necesita que cada
  sujeto tenga todos los agarres; si no, su media mezclaría agarres distintos y sesgaría los centroides.

El sujeto se identifica con el ID compuesto `subject + origin_db`. Si hay varias filas por sujeto × agarre
(ventanas), **no se colapsan**: todas se usan, pero cada sujeto pesa lo mismo en los centroides.

In [26]:
df, info = cargar_datos(DATA_PATH, FEATURE_COLS)
FEATURE_COLS = info['feature_cols']
X = df[FEATURE_COLS].values.astype(float)
y = df[LABEL_COL].values
groups = df['group'].values.astype(object)
DB_OF_GROUP = df.groupby('group')[DB_COL].first().to_dict()

STIMULI = sorted(np.unique(y).tolist())
LABELS = [f'Agarre {s}' for s in STIMULI]
N_STIM = len(STIMULI)
N_FEATURES = len(FEATURE_COLS)
N_SUBJ = len(DB_OF_GROUP)

FOLD_MAP = asignar_folds(groups, N_FOLDS, RANDOM_STATE)
folds = np.array([FOLD_MAP[g] for g in groups])

DATASET_TAG = f'{Path(DATA_PATH).stem}__{N_FEATURES}f__{DISTANCIA}'
SAVE_DIR = f'../figures/jerarquico/{DATASET_TAG}'
os.makedirs(SAVE_DIR, exist_ok=True)

filas_celda = df.groupby(['group', LABEL_COL]).size()

print(f'Dataset: {DATA_PATH}')
print(f'Features: {N_FEATURES}')
print(f'Distancia: {DISTANCIA} | enlaces: {LINKAGE_METHODS} | centrado por sujeto: siempre')
print(f'Figuras: {SAVE_DIR}')
print(f'Filas eliminadas — reposo: {info["n_rest"]} | con NaN: {info["n_nan"]} | '
      f'sujetos con agarres faltantes: {len(info["incompletos"])} {info["incompletos"][:10]}')
print(f'Filas: {len(df)} | agarres ({N_STIM}): {STIMULI} | sujetos: {N_SUBJ}')
print('Sujetos por base de datos:', pd.Series(DB_OF_GROUP).value_counts().sort_index().to_dict())
if filas_celda.max() == 1:
    print('Filas por sujeto × agarre: 1 (sin ventanas)')
else:
    print(f'Filas por sujeto × agarre: {filas_celda.min()}–{filas_celda.max()} '
          f'(media {filas_celda.mean():.1f}) → ventanas como filas; el sujeto sigue siendo la unidad')
if N_FEATURES > N_SUBJ:
    print(f'AVISO: {N_FEATURES} variables para {N_SUBJ} sujetos; Ledoit-Wolf encogerá mucho la covarianza.')

Dataset: ../data/master/df_stats_wnd_mean.csv
Features: 42
Distancia: crossnobis | enlaces: ['average', 'complete'] | centrado por sujeto: siempre
Figuras: ../figures/jerarquico/df_stats_wnd_mean__42f__crossnobis
Filas eliminadas — reposo: 0 | con NaN: 0 | sujetos con agarres faltantes: 0 []
Filas: 609 | agarres (7): [22, 25, 27, 30, 31, 37, 40] | sujetos: 87
Sujetos por base de datos: {'NinaProDB1': 27, 'NinaProDB2': 40, 'NinaProDB4': 10, 'NinaProDB5': 10}
Filas por sujeto × agarre: 1 (sin ventanas)


## 2. Efecto sujeto y base de datos

**Por qué el centrado por sujeto es obligatorio.** Cada sujeto aporta todos los agarres (medidas
repetidas). Si un sujeto tiene la señal más alta en todos sus agarres, ese desplazamiento no dice nada de
las diferencias entre agarres, pero entra en la covarianza dentro de los agarres, la infla y aplana las
distancias. Además, cada sujeto pertenece a **una sola base de datos**, así que las diferencias entre DB
(electrodos, protocolo, amplificador) son, en el fondo, desplazamientos de sujeto. Si no se quitan, el
análisis separaría por DB en lugar de por agarre.

**Centrado:** a cada fila se le resta la media de su sujeto, calculada como la media de sus medias por
agarre, para que los agarres con más ventanas no pesen más. Con el diseño completo, las diferencias entre
los centroides de los agarres **no cambian**; solo se limpia la covarianza.

**Comprobación:** sobre las medias por sujeto × agarre, se reparte la varianza de cada variable en:
- **DB**: diferencias entre bases de datos.
- **Sujeto (dentro de DB)**: diferencias entre sujetos de la misma DB.
- **Agarre**: lo que interesa.
- **Residual**: interacción sujeto × agarre (cada sujeto hace los agarres a su manera).

Después del centrado, DB y sujeto deben quedar en **0 %**.

In [27]:
var_antes = descomponer_varianza(X, groups, y)
Xd = centrar_por_sujeto(X, groups, y)        # los datos con los que se miden las distancias
var_despues = descomponer_varianza(Xd, groups, y)

resumen_varianza = pd.DataFrame({'antes (media %)': var_antes.mean(),
                                 'después (media %)': var_despues.mean()})
display(resumen_varianza.round(2))
top_db = var_antes['DB'].sort_values(ascending=False).head(5)
print('Variables con más varianza de DB antes del centrado:',
      ', '.join(f'{k} ({v:.1f} %)' for k, v in top_db.items()))

,antes (media %),después (media %)
DB,6.04,0.00
Sujeto (dentro de DB),12.70,0.00
Agarre,30.61,36.95
Residual (sujeto × agarre),50.65,63.05


Variables con más varianza de DB antes del centrado: VAR_STD_min (26.1 %), VAR_min (20.5 %), Max_first (15.2 %), Max_min (13.6 %), Energy_min (13.4 %)


## 3. Covarianza Ledoit-Wolf y distancia entre centroides

**Por qué Mahalanobis y no euclídea:** las variables de amplitud están muy correlacionadas entre sí. En la
distancia euclídea, una misma información contaría tantas veces como variables redundantes haya.
Mahalanobis lo corrige con la covarianza: las variables redundantes cuentan una sola vez, y una diferencia
pesa más si ocurre en una dirección en la que los datos suelen variar poco.

**Qué covarianza se usa:** la covarianza **dentro de los agarres** (*pooled*), calculada sobre los datos
centrados por sujeto: a cada fila se le resta el centroide de su agarre y se calcula la covarianza de esos
residuos. Con una fila por sujeto × agarre, mide cómo varían los sujetos dentro de un agarre. Con ventanas
como filas, mide el **ruido ventana a ventana**, como en el crossnobis de RSA. En ese caso la escala de las
distancias es la de ventana: no se compara en valor absoluto con un dataset de una fila por sujeto, pero sí
el orden de los pares y el dendrograma.

**Por qué Ledoit-Wolf:** con variables casi colineales, la covarianza empírica es casi singular y su inversa
es inestable. Ledoit-Wolf la "encoge" hacia una matriz diagonal en la cantidad óptima (`shrinkage`).

**Mahalanobis vs crossnobis (`DISTANCIA`):** los centroides tienen error de muestreo. En la Mahalanobis
clásica ese error se eleva al cuadrado y **siempre suma**, así que la distancia sale sesgada hacia arriba,
sobre todo en los pares más parecidos. Crossnobis divide a los sujetos en `N_FOLDS` grupos y multiplica la
diferencia de centroides de un fold por la de **otro** fold. Como el ruido de folds independientes no
correlaciona, se cancela, y d² es **insesgada**. Puede salir ligeramente negativa cuando la distancia real
es ≈ 0; en ese caso se recorta a 0 para el dendrograma.

In [28]:
D, d2, lw = distancia(Xd, y, groups, folds)
cov_emp = np.cov(Xd - class_centroids(Xd, y, groups)[np.searchsorted(STIMULI, y)], rowvar=False)

print(f'Shrinkage Ledoit-Wolf: {lw.shrinkage_:.3f}  (0 = covarianza empírica, 1 = totalmente diagonal)')
print(f'Número de condición — empírica: {np.linalg.cond(cov_emp):,.0f} | '
      f'Ledoit-Wolf: {np.linalg.cond(lw.covariance_):,.0f}')
iu = np.triu_indices(N_STIM, 1)
if DISTANCIA == 'crossnobis':
    print(f'Pares con d² crossnobis < 0 (distancia real ≈ 0): {(d2[iu] < 0).sum()} de {len(iu[0])}')

D_df = pd.DataFrame(D, index=LABELS, columns=LABELS)
pairs_df = (pd.DataFrame([{'agarre_a': STIMULI[i], 'agarre_b': STIMULI[j], 'distancia': D[i, j]}
                          for i, j in zip(*iu)])
            .sort_values('distancia').reset_index(drop=True))
pairs_df.index = pairs_df.index + 1
print('\nPares más parecidos (menor distancia):')
display(pairs_df.head(5).round(3))
print('Pares más distintos (mayor distancia):')
display(pairs_df.tail(3).round(3))

Shrinkage Ledoit-Wolf: 0.009  (0 = covarianza empírica, 1 = totalmente diagonal)
Número de condición — empírica: 1,498,727 | Ledoit-Wolf: 2,369
Pares con d² crossnobis < 0 (distancia real ≈ 0): 0 de 21

Pares más parecidos (menor distancia):


,agarre_a,agarre_b,distancia
1,25,27,1.674
2,22,30,1.890
3,22,27,1.971
4,25,30,2.277
5,30,31,2.345


Pares más distintos (mayor distancia):


,agarre_a,agarre_b,distancia
19,22,40,5.582
20,27,40,5.931
21,25,40,5.960


## 4. Dendrograma: elección del método de enlace

El dendrograma une primero los agarres más parecidos y luego va juntando grupos. La altura de cada unión es
la distancia a la que se juntan: uniones bajas = agarres muy parecidos.

**Método de enlace** (cómo se mide la distancia entre dos grupos ya formados):
- `average`: media de las distancias entre sus miembros (UPGMA). Es el más usado con matrices de distancia.
- `complete`: la distancia máxima entre sus miembros.
- `ward`: minimiza la varianza dentro de los grupos. Solo es válido con **Mahalanobis** (que es euclídea en
  el espacio "blanqueado"); con crossnobis se excluye. Sus alturas no son distancias, sino incrementos de
  varianza.

Se elige el método con mayor **correlación cofenética** (cuánto respeta el dendrograma las distancias
originales). Con pocos agarres hay pocos pares (21 con 7 agarres), así que las diferencias pequeñas no son
concluyentes: si hay empate (±0.001) gana el primero de `LINKAGE_METHODS`. La estabilidad de la sección 5 es
más informativa que la cofenética.

In [29]:
coph, LINKAGE, empatados = elegir_enlace(D, LINKAGE_METHODS)
display(pd.Series(coph, name='correlación cofenética').to_frame().round(3))
if len(empatados) > 1:
    print(f'Empate entre {empatados} → se usa {LINKAGE} (primero de LINKAGE_METHODS)')
print(f'Método elegido: {LINKAGE} (cofenética = {coph[LINKAGE]:.3f})')

Z = linkage(squareform(D, checks=False), method=LINKAGE)
ORDER = dendrogram(Z, no_plot=True, labels=LABELS)['ivl']
plot_dist_heatmap(D_df, ORDER)

,correlación cofenética
average,0.957
complete,0.954


Método elegido: average (cofenética = 0.957)


## 5. Estabilidad del dendrograma (bootstrap por sujetos)

Un dendrograma siempre se puede dibujar, aunque las diferencias sean ruido. Para saber qué uniones son
fiables, se repite todo el proceso `N_BOOT` veces (centroides, covarianza Ledoit-Wolf, distancias y
dendrograma) con una muestra de **sujetos** tomada **con reemplazo**, cada uno con todas sus filas:
- El remuestreo se hace **dentro de cada base de datos**, para que todas las muestras mantengan la mezcla de
  DB del original.
- Un sujeto que sale dos veces cuenta como dos sujetos.
- En crossnobis, las dos copias se quedan en el mismo fold; si cayeran en folds distintos, volvería el sesgo.

Resultados:
- **Soporte de cada rama** (`dendrograma.pdf`): % de remuestreos en que aparece exactamente ese grupo de
  agarres. Referencia: ≥ 70 % es una agrupación sólida y < 50 % es poco fiable. Exige que el grupo
  aparezca *exacto*, así que es un criterio conservador.
- **IC 95 % de las distancias** (tabla `pairs_df`): percentiles 2.5–97.5 de cada distancia,
  **corregidos por sesgo**. Al duplicar sujetos hay menos sujetos distintos para estimar la covarianza;
  con muchas variables por sujeto, la covarianza se subestima y las distancias del bootstrap salen
  infladas. Por eso se resta a cada par su sesgo medio (media bootstrap − estimación), y se imprime
  cuánto vale.

In [30]:
support, D_boot, n_ok = bootstrap_sujetos(Xd, y, groups, folds, Z)
# IC percentil corregido por sesgo: con muchas variables por sujeto, duplicar sujetos subestima la
# covarianza y las distancias del bootstrap salen infladas → se resta el sesgo medio de cada par.
bias = D_boot.mean(axis=0) - D
D_lo, D_hi = np.percentile(D_boot, [2.5, 97.5], axis=0)
D_lo, D_hi = np.clip(D_lo - bias, 0, None), np.clip(D_hi - bias, 0, None)
rel_bias = 100 * bias[iu] / np.where(D[iu] > 0, D[iu], np.nan)
print(f'Remuestreos válidos: {n_ok} de {N_BOOT}')
print(f'Sesgo medio del bootstrap: {np.nanmean(rel_bias):+.1f} % de la distancia (restado en los IC)')
if np.nanmean(np.abs(rel_bias)) > 5:
    print('  AVISO: sesgo > 5 % — típico con muchas variables por sujeto; los IC se corrigen, pero el '
          'soporte de ramas puede ser algo optimista/pesimista.')

support_df = (pd.DataFrame([{'grupo': ' + '.join(str(STIMULI[i]) for i in sorted(c)),
                             'n_agarres': len(c), 'soporte_%': s} for c, s in support.items()])
              .sort_values(['n_agarres', 'soporte_%'], ascending=[True, False]).reset_index(drop=True))
display(support_df.round(1))

idx = {s: i for i, s in enumerate(STIMULI)}
pairs_df['IC95_inf'] = [D_lo[idx[a], idx[b]] for a, b in zip(pairs_df.agarre_a, pairs_df.agarre_b)]
pairs_df['IC95_sup'] = [D_hi[idx[a], idx[b]] for a, b in zip(pairs_df.agarre_a, pairs_df.agarre_b)]
pairs_df['sesgo_bootstrap'] = [bias[idx[a], idx[b]] for a, b in zip(pairs_df.agarre_a, pairs_df.agarre_b)]

display(pairs_df.round(3))

plot_dendrograma(Z, support, n_ok)

Remuestreos válidos: 500 de 500
Sesgo medio del bootstrap: +3.5 % de la distancia (restado en los IC)


,grupo,n_agarres,soporte_%
0,25 + 27,2,90.0
1,22 + 30,2,84.0
2,22 + 25 + 27 + 30,4,90.0
3,22 + 25 + 27 + 30 + 31,5,97.2
4,22 + 25 + 27 + 30 + 31 + 37,6,100.0


,agarre_a,agarre_b,distancia,IC95_inf,IC95_sup,sesgo_bootstrap
1,25,27,1.674,1.283,2.052,0.076
2,22,30,1.890,1.457,2.285,0.050
3,22,27,1.971,1.609,2.353,0.064
4,25,30,2.277,1.874,2.674,0.078
5,30,31,2.345,2.022,2.737,0.076
6,22,25,2.456,2.114,2.845,0.089
7,27,30,2.499,2.187,2.824,0.086
8,25,31,2.604,2.118,3.091,0.108
9,22,31,2.886,2.421,3.332,0.107
10,22,37,2.981,2.629,3.344,0.093


## 6. Sensibilidad: Mahalanobis vs crossnobis

Se calculan las dos distancias sobre los mismos datos centrados y se comparan par a par:
- **Cambio medio negativo:** crossnobis da distancias menores, que es el sesgo que corrige. Debería
  notarse más en los pares cercanos.
- Si el **orden** de los pares (correlación de Spearman) se mantiene, las familias del dendrograma no
  dependen de la distancia elegida.

In [31]:
var_df = pd.DataFrame({m: distancia(Xd, y, groups, folds, m)[0][iu]
                       for m in ['mahalanobis', 'crossnobis']},
                      index=[f'{STIMULI[i]}-{STIMULI[j]}' for i, j in zip(*iu)])

print(f"Cambio medio crossnobis vs Mahalanobis: "
      f"{100 * (var_df['crossnobis'] / var_df['mahalanobis'] - 1).mean():+.1f} %")
print(f"Spearman entre ambas: {var_df.corr(method='spearman').iloc[0, 1]:.3f}")
display(var_df.sort_values('mahalanobis').head(5).round(3))

Cambio medio crossnobis vs Mahalanobis: -4.1 %
Spearman entre ambas: 1.000


,mahalanobis,crossnobis
25-27,1.887,1.674
22-30,2.049,1.890
22-27,2.120,1.971
25-30,2.416,2.277
30-31,2.442,2.345


## 7. Resumen

In [32]:
print(f'Configuración: {N_FEATURES} variables, {N_SUBJ} sujetos, {DISTANCIA} con centrado por sujeto, '
      f'enlace {LINKAGE} (cofenética {coph[LINKAGE]:.3f}), shrinkage LW {lw.shrinkage_:.3f}, '
      f'{n_ok} remuestreos')
print(f"Varianza de DB + sujeto: {var_antes[['DB', 'Sujeto (dentro de DB)']].sum(axis=1).mean():.1f} % "
      f"antes → {var_despues[['DB', 'Sujeto (dentro de DB)']].sum(axis=1).mean():.1f} % después del centrado\n")
print('Pares más parecidos (IC 95 % bootstrap):')
for r in pairs_df.head(3).itertuples():
    print(f'  {r.agarre_a}–{r.agarre_b}: {r.distancia:.2f} [{r.IC95_inf:.2f}, {r.IC95_sup:.2f}]')
print('Pares más distintos:')
for r in pairs_df.tail(3).itertuples():
    print(f'  {r.agarre_a}–{r.agarre_b}: {r.distancia:.2f} [{r.IC95_inf:.2f}, {r.IC95_sup:.2f}]')
print('\nRamas con soporte ≥ 70 %:')
solid = support_df[support_df['soporte_%'] >= 70]
print('  ' + ('\n  '.join(f'{{{r.grupo}}} ({r["soporte_%"]:.0f} %)' for _, r in solid.iterrows())
              if len(solid) else 'ninguna'))
print(f'\nFiguras en {SAVE_DIR}: {sorted(os.listdir(SAVE_DIR))}')

Configuración: 42 variables, 87 sujetos, crossnobis con centrado por sujeto, enlace average (cofenética 0.957), shrinkage LW 0.009, 500 remuestreos
Varianza de DB + sujeto: 18.7 % antes → 0.0 % después del centrado

Pares más parecidos (IC 95 % bootstrap):
  25–27: 1.67 [1.28, 2.05]
  22–30: 1.89 [1.46, 2.28]
  22–27: 1.97 [1.61, 2.35]
Pares más distintos:
  22–40: 5.58 [5.13, 6.13]
  27–40: 5.93 [5.47, 6.49]
  25–40: 5.96 [5.42, 6.53]

Ramas con soporte ≥ 70 %:
  {25 + 27} (90 %)
  {22 + 30} (84 %)
  {22 + 25 + 27 + 30} (90 %)
  {22 + 25 + 27 + 30 + 31} (97 %)
  {22 + 25 + 27 + 30 + 31 + 37} (100 %)

Figuras en ../figures/jerarquico/df_stats_wnd_mean__42f__crossnobis: ['dendrograma.pdf', 'distancia_heatmap.pdf']


## 8. Interpretación

*(Completar con los resultados.)*

- **Efecto sujeto/DB (sección 2):** cuánta varianza de DB y de sujeto había antes del centrado. Si era
  alta en alguna variable, sin centrar habría dominado el análisis.
- **Familias de agarres:** ramas con soporte alto (≥ 70 %) en el dendrograma y bloques de distancias
  bajas en el heatmap (por ejemplo, agarres de mucha fuerza frente a agarres de precisión).
- **Contraste con la separabilidad (notebook 07):** los pares más cercanos aquí deberían coincidir con los
  pares más confundidos de la LDA en LOSO. Si coinciden, las dos vistas (supervisada y de distancias)
  cuentan la misma historia.
- **Sensibilidad (sección 6):** si el orden de los pares se mantiene entre Mahalanobis y crossnobis, las
  familias no dependen de la distancia elegida.
- **Ramas con soporte bajo:** el orden de esas uniones depende de qué sujetos entren en la muestra; no
  conviene interpretarlas.

**Limitaciones**
- Al centrar por sujeto se pierde el **nivel absoluto** de cada sujeto: dos agarres que solo se
  diferenciaran en el nivel general del sujeto no se distinguen. Es intencionado: ese nivel mezcla al
  sujeto con su base de datos.
- La distancia supone una covarianza común a todos los agarres (pooled).
- Con ventanas como filas, la escala de las distancias es la del ruido de ventana: el valor absoluto no se
  compara con datasets de una fila por sujeto, pero el orden de los pares y el dendrograma sí.
- El soporte bootstrap mide la estabilidad frente al muestreo de sujetos, no frente a la elección de
  variables.

In [33]:
sello.mostrar()

### Sello de procedencia

| Campo | Valor |
|---|---|
| Notebook | `09_clustering_analysis.ipynb` |
| Commit | `07165f8` |
| Rama | `main` |
| Estado | sucio (5 archivos sin commitear) |
| Ejecutado | 2026-10-15T12:01:35 |

**Archivos generados en esta ejecución:**

- `figures/jerarquico/df_stats_wnd_mean__42f__crossnobis/dendrograma.pdf`
- `figures/jerarquico/df_stats_wnd_mean__42f__crossnobis/distancia_heatmap.pdf`

> Guardado en `PROVENANCE.md` (raíz del repo).